In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
import logging
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, f1_score, roc_auc_score

from src.helper import *
from src.model_sota import *

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#set config
DATA_PATH = "data/deep-detect/dataset/"
# EMBEDDED_DATA_PATH = "output/embeddings/wav2vec2/"
OUTPUT_PATH = "output/"
PREDS_PATH = "output/preds/"
MODELS_PATH = "models/"

#model training config
BATCH_SIZE = 16
EPOCHS = 20
# EPOCHS = 2
LEARNING_RATE = 0.0001

In [ ]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)
if not os.path.exists(PREDS_PATH):
    os.mkdir(PREDS_PATH)
if not os.path.exists(MODELS_PATH):
    os.mkdir(MODELS_PATH)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_05_sota_model_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting SOTA modeling...")

### Preparing the data loader

In [ ]:
logger.info(f"preparing the dataloader...")

train_ds = AudioFolderDataset(os.path.join(DATA_PATH, "training/"))
test_ds = AudioFolderDataset(os.path.join(DATA_PATH, "testing/"))
holdout_ds = AudioFolderDataset(os.path.join(DATA_PATH, "holdout/"))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Inspect one batch
for X, y, path in train_loader:
    logger.info(f"batch input shape: {X.shape}")
    logger.info(f"batch labels shape: {y.shape}")
    break

### Testing SOTA HybridAudioClassifier Model 

It mixes a strong CNN backbone (PANNs-style), residual + squeeze-and-excitation blocks, and a Transformer encoder on top (AST / PaSST style). This hybrid approach is widely used in modern audio classification and often gives top results on benchmarks like AudioSet, ESC-50 and Speech Commands.

SpecAugment
- Works on spectrograms (time–frequency images). It randomly masks:
	- time segments → makes the model robust to shifts in speech/audio tempo.
	- frequency bands → forces the model to not depend on narrow frequency cues.
- This helps generalization (especially with limited data).


In [ ]:
logger.info(f"starting HybridAudioClassifier training...")
logger.info(f"learning rate: {LEARNING_RATE}, epoch: {EPOCHS}, batch size: {BATCH_SIZE}")

model = HybridAudioClassifier(num_classes=2, use_spec_augment=True).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

model_name = f"hybrid_audio_classifier__cross_entropy__adamw__lr_{str(LEARNING_RATE).replace('.', '_')}__epochs_{EPOCHS}__bs_{BATCH_SIZE}"
model_save_path = os.path.join(MODELS_PATH, f"{model_name}.pth")

train_losses = []
train_f1s = []
val_losses = []
val_f1s = []

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device, use_mixup=False)
    val_loss, val_f1 = evaluate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    train_f1s.append(train_f1)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)

    scheduler.step()

    logger.info(f"Epoch {epoch}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} f1: {train_f1:.4f} | "
            f"Val loss: {val_loss:.4f} f1: {val_f1:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), model_save_path)

logger.info(f"HybridAudioClassifier training finished")

In [ ]:
df_train_history = pd.DataFrame({
    'epoch': range(1, len(train_losses)+1),
    'train_losses': train_losses,
    'train_f1s': train_f1s,
    'val_losses': val_losses,
    'val_f1s': val_f1s,
})

train_history_save_path = os.path.join(OUTPUT_PATH, "train_history/")
if not os.path.exists(train_history_save_path):
    os.mkdir(train_history_save_path)

df_train_history.to_csv(os.path.join(train_history_save_path, f"{model_name}_train_history.csv"), index=False)
logger.info(f"{model_name} train history result saved to {train_history_save_path}")

In [ ]:
#plotting training results

plt.plot(range(1, len(train_losses)+1), train_losses, 'b')
plt.plot(range(1, len(val_losses)+1), val_losses, 'r')
plt.xlabel("Number of epochs")
plt.ylabel("Loss")
plt.title("Loss vs Number of epochs")
plt.legend(['train', 'validation'])
plt.show()

plt.plot(range(1, len(train_f1s)+1), train_f1s, 'b')
plt.plot(range(1, len(val_f1s)+1), val_f1s, 'r')
plt.xlabel("Number of epochs")
plt.ylabel("F1-Score")
plt.title("F1-Score vs Number of epochs")
plt.legend(['train', 'validation'])
plt.show()

In [ ]:
logger.info(f"evaluating {model_name} best performing epoch...")
model.load_state_dict(torch.load(model_save_path))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for X_batch, y_batch, path_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = outputs.argmax(dim=1).cpu().numpy()
        y_true.extend(y_batch.numpy())
        y_pred.extend(preds)

#get score summary
logger.info(f"{model_name} test f1-score: {f1_score(y_true, y_pred)}")
logger.info(f"{model_name} classification report: \n{classification_report(y_true, y_pred, digits=5)}")

In [ ]:
logger.info(f"generating {model_name} holdout prediction...")
model.eval()
y_holdout_id = []
y_holdout_pred = []
with torch.no_grad():
    for X_batch, y_batch, path_batch in holdout_loader:
        id = [p.split("/")[-1].split(".")[0]+".wav" for p in path_batch]
        y_holdout_id.extend(id)

        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = outputs.argmax(dim=1).cpu().numpy()
        y_holdout_pred.extend(preds)

preds_save_path = os.path.join(PREDS_PATH, f'{model_name}_preds.csv')
df_holdout_preds = pd.DataFrame()
df_holdout_preds['id'] = y_holdout_id
df_holdout_preds['label'] = y_holdout_pred
df_holdout_preds.to_csv(preds_save_path, index=False)
logger.info(f"{model_name} holdout preds saved to {preds_save_path}")

### Applying Mixup

Mixup
- Takes two examples (x₁, y₁) and (x₂, y₂) and blends them:
- x~=λx1+(1−λ)x2
- y~=λy1+(1−λ)y2
- This encourages the model to learn smooth decision boundaries instead of memorizing

In [ ]:
logger.info(f"starting HybridAudioClassifier + Mixup training...")
logger.info(f"learning rate: {LEARNING_RATE}, epoch: {EPOCHS}, batch size: {BATCH_SIZE}")

model = HybridAudioClassifier(num_classes=2, use_spec_augment=True).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

model_name = f"hybrid_audio_classifier_mixup__cross_entropy__adamw__lr_{str(LEARNING_RATE).replace('.', '_')}__epochs_{EPOCHS}__bs_{BATCH_SIZE}"
model_save_path = os.path.join(MODELS_PATH, f"{model_name}.pth")

train_losses = []
train_f1s = []
val_losses = []
val_f1s = []

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device, use_mixup=True)
    val_loss, val_f1 = evaluate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    train_f1s.append(train_f1)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)

    scheduler.step()

    logger.info(f"Epoch {epoch}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} f1: {train_f1:.4f} | "
            f"Val loss: {val_loss:.4f} f1: {val_f1:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), model_save_path)

logger.info(f"HybridAudioClassifier + Mixup training finished")